**1. Load Dataset**

In [1]:
import pandas as pd 
df = pd.read_csv("../data/cars_dataframe.csv")
df.head() #prints the first 5 rows of the dataframe

,Brand,Model,Year,Condition,Mileage,Gearbox,Fiscal Power,Fuel,Equipment,Number of Doors,Origin,First Owner,Location,Sector,Price
0,Volkswagen,Polo,1997,Good,200 000 - 249 999,Manual,8 CV,Diesel,"['Leather Seats', 'CD/MP3/Bluetooth']",NaN,NaN,NaN,Tétouan,City Center,27000.0
1,Dacia,Sandero,2012,Very Good,180 000 - 189 999,Manual,6 CV,Diesel,"['Rear Camera', 'Electric Windows']",5.0,WW in Morocco,No,Temara,City Center,69000.0
2,Volkswagen,Polo,2004,Fair,300 000 - 349 999,Manual,8 CV,Diesel,"['Alloy Wheels', 'Electric Windows']",5.0,WW in Morocco,No,Safi,Chenguite,45000.0
3,Bentley,Continental gtc,2009,Excellent,45 000 - 49 999,Automatic,34 CV,Petrol,"['Rear Camera', 'Electric Windows']",3.0,Customs-cleared car,Yes,Casablanca,Aïn Chock,720000.0
4,mini,cooper,2013,Excellent,85 000 - 89 999,Automatic,8 CV,Diesel,"['Airbags', 'CD/MP3/Bluetooth', 'Rear Camera',...",5.0,WW in Morocco,No,Fès,Agdal,128000.0


**2. Explore Data**

- This is where we inspect the data, understand columns names, data types, find missing values etc before we move to cleaning. 

In [2]:
df.shape #number of rows and columns in dataframe

(101896, 15)

In [3]:
df.columns #column names

Index(['Brand', 'Model', 'Year', 'Condition', 'Mileage', 'Gearbox',
       'Fiscal Power', 'Fuel', 'Equipment', 'Number of Doors', 'Origin',
       'First Owner', 'Location', 'Sector', 'Price'],
      dtype='str')

In [4]:
df.describe() #stats of the dataframe

,Number of Doors,Price
count,90481.000000,7.641300e+04
mean,4.867243,4.980144e+05
std,0.497888,1.587622e+07
min,3.000000,1.000000e+00
25%,5.000000,6.000000e+04
50%,5.000000,1.100000e+05
75%,5.000000,1.700000e+05
max,5.000000,1.600000e+09


In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 101896 entries, 0 to 101895
Data columns (total 15 columns):
 #   Column           Non-Null Count   Dtype  
---  ------           --------------   -----  
 0   Brand            101895 non-null  str    
 1   Model            101895 non-null  str    
 2   Year             101896 non-null  str    
 3   Condition        98313 non-null   str    
 4   Mileage          101895 non-null  str    
 5   Gearbox          101896 non-null  str    
 6   Fiscal Power     101616 non-null  str    
 7   Fuel             101896 non-null  str    
 8   Equipment        101896 non-null  str    
 9   Number of Doors  90481 non-null   float64
 10  Origin           89921 non-null   str    
 11  First Owner      87196 non-null   str    
 12  Location         101896 non-null  str    
 13  Sector           101650 non-null  str    
 14  Price            76413 non-null   float64
dtypes: float64(2), str(13)
memory usage: 11.7 MB


In [6]:
df.isnull().sum() #how many null values in each column

Brand                  1
Model                  1
Year                   0
Condition           3583
Mileage                1
Gearbox                0
Fiscal Power         280
Fuel                   0
Equipment              0
Number of Doors    11415
Origin             11975
First Owner        14700
Location               0
Sector               246
Price              25483
dtype: int64

Since Price is the target variable, rows with missing prices cannot be used for training, and will be discarded. 

In [7]:
df[['Year', 'Mileage', 'Fiscal Power', 'Price']].head(10)

,Year,Mileage,Fiscal Power,Price
0,1997,200 000 - 249 999,8 CV,27000.0
1,2012,180 000 - 189 999,6 CV,69000.0
2,2004,300 000 - 349 999,8 CV,45000.0
3,2009,45 000 - 49 999,34 CV,720000.0
4,2013,85 000 - 89 999,8 CV,128000.0
5,2020,200 000 - 249 999,8 CV,NaN
6,2015,0 - 4 999,6 CV,NaN
7,2015,200 000 - 249 999,8 CV,220000.0
8,2018,30 000 - 34 999,8 CV,165000.0
9,2000,25 000 - 29 999,7 CV,40000.0


Some columns should be modified so their values are more appropriate, like the Mileage, which is written as a range but should be converted to a numerical value by taking the midpoint or average. 

**3. Preprocessing**

In [8]:
df_clean = df.copy() #creating a copy of the df so we dont modify the original dataset

In [9]:
#remove the rows where price is missing
df_clean = df_clean.dropna(subset=["Price"])

In [10]:
df_clean.shape
#Now we only have 76k rows after discarding rows with missing price

(76413, 15)

In [11]:
df_clean["Year"] = pd.to_numeric(df_clean["Year"], errors="coerce")
df_clean["Year"].dtype

dtype('float64')

In [12]:
#how many null values in Year column after converting to numeric 
df_clean["Year"].isnull().sum() 

np.int64(187)

In [13]:
#Since we have 76k rows, dropping only 187 rows is fine
df_clean = df_clean.dropna(subset = ["Year"])

In [14]:
#convert year to int
df_clean["Year"] = df_clean["Year"].astype(int) 

In [15]:
df_clean["Year"].dtype
df_clean.shape

(76226, 15)

In [16]:
#Next, we will fix fiscal power so that 8 CV becomes 8. (remove "CV")
df_clean["Fiscal Power"] = (
    df_clean["Fiscal Power"]
    .str.replace(" CV", "", regex=False) )

In [17]:
#Convert fiscal power to numeric
df_clean["Fiscal Power"] = pd.to_numeric(df_clean["Fiscal Power"], errors="coerce")     

In [18]:
df_clean["Fiscal Power"].isnull().sum()

np.int64(417)

In [19]:
#remove the null values in fiscal power column  
df_clean = df_clean.dropna(subset = ["Fiscal Power"]) 

In [20]:
df_clean["Fiscal Power"] = df_clean["Fiscal Power"].astype(int)
df_clean.shape

(75809, 15)

In [21]:
#Convert mileage to a single number by taking the midpoint of the range
import numpy as np

def mileage_to_midpoint(value):
    if pd.isna(value):
        return np.nan
    
    value = str(value).replace(" ", "")
    
    if "-" not in value:
        return np.nan
    
    start, end = value.split("-")
    return (int(start) + int(end)) / 2

In [22]:
df_clean["Mileage"] = df_clean["Mileage"].apply(mileage_to_midpoint)

In [23]:
df_clean["Mileage"].isnull().sum()
df_clean = df_clean.dropna(subset=["Mileage"])
df_clean["Mileage"] = df_clean["Mileage"].astype(int)

In [24]:
df_clean.shape

(75123, 15)

In [25]:
#Now, we will clean the categorical columns
df_clean[["Brand", "Model", "Fuel", "Gearbox", "Location"]].head()

,Brand,Model,Fuel,Gearbox,Location
0,Volkswagen,Polo,Diesel,Manual,Tétouan
1,Dacia,Sandero,Diesel,Manual,Temara
2,Volkswagen,Polo,Diesel,Manual,Safi
3,Bentley,Continental gtc,Petrol,Automatic,Casablanca
4,mini,cooper,Diesel,Automatic,Fès


In [26]:
categorical_cols = ["Brand", "Model", "Fuel", "Gearbox", "Location"]

for col in categorical_cols:
    print(col)
    print(df_clean[col].nunique())
    print(df_clean[col].unique()[:10])
    print("-" * 30)

Brand
73
<StringArray>
['Volkswagen',      'Dacia',    'Bentley',       'mini',       'Audi',
      'Honda',    'Peugeot',    'Citroen',    'Renault',    'Hyundai']
Length: 10, dtype: str
------------------------------
Model
946
<StringArray>
[           'Polo',         'Sandero', 'Continental gtc',          'cooper',
              'A3',           'Caddy',            'HR-V',             '301',
          'GOLF 5',             '307']
Length: 10, dtype: str
------------------------------
Fuel
5
<StringArray>
['Diesel', 'Petrol', 'Electrique', 'Hybrid', 'LPG']
Length: 5, dtype: str
------------------------------
Gearbox
2
<StringArray>
['Manual', 'Automatic']
Length: 2, dtype: str
------------------------------
Location
376
<StringArray>
[   'Tétouan',     'Temara',       'Safi', 'Casablanca',        'Fès',
  'Marrakech',     'Agadir',     'Martil',       'Salé', 'Mohammedia']
Length: 10, dtype: str
------------------------------


In [27]:
# Capitalize the first letter of each word in the categorical columns
text_cols = ["Brand", "Model", "Fuel", "Gearbox", "Location"]

for col in text_cols:
    df_clean[col] = df_clean[col].str.strip().str.title()

In [28]:
df_clean[text_cols].head()

,Brand,Model,Fuel,Gearbox,Location
0,Volkswagen,Polo,Diesel,Manual,Tétouan
1,Dacia,Sandero,Diesel,Manual,Temara
2,Volkswagen,Polo,Diesel,Manual,Safi
3,Bentley,Continental Gtc,Petrol,Automatic,Casablanca
4,Mini,Cooper,Diesel,Automatic,Fès


Next, we fix invalid values in the categorical columns. Some cities are written in Arabic (for example الدار البيضاء is Casablanca), and one row has a mileage range in the Fuel column.

In [29]:
#Replace the Arabic city names with the spelling used in the rest of the data
arabic_cities = {
    "أكادير": "Agadir", "الجديدة": "El Jadida", "الدار البيضاء": "Casablanca",
    "الرباط": "Rabat", "القنيطرة": "Kénitra", "المحمدية": "Mohammedia",
    "الناظور": "Nador", "تطوان": "Tétouan", "تمارة": "Temara", "طنجة": "Tanger",
    "فاس": "Fès", "مراكش": "Marrakech", "مكناس": "Meknès", "وجدة": "Oujda"
}
df_clean["Location"] = df_clean["Location"].replace(arabic_cities)

In [30]:
#Fuel should only contain real fuel types, so we print them to check
df_clean["Fuel"].value_counts()

Fuel
Diesel        59977
Petrol        14239
Hybrid          657
Electrique      206
Lpg              44
Name: count, dtype: int64

In [31]:
#Remove the row where Fuel contains a mileage range instead of a fuel type
valid_fuels = ["Diesel", "Petrol", "Hybrid", "Electrique", "Lpg"]
df_clean = df_clean[df_clean["Fuel"].isin(valid_fuels)]

**Handling price outliers**: the lowest price is 1 MAD and the highest is 1.6 billion MAD, which are placeholders or typing mistakes. A model trained on them performs very badly (our first run had an MAE of about 800,000 MAD), so we remove them.

In [32]:
df_clean["Price"].describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99])

count    7.512300e+04
mean     4.726864e+05
std      1.472586e+07
min      1.000000e+00
1%       1.070000e+04
5%       2.700000e+04
50%      1.100000e+05
95%      3.650000e+05
99%      7.500000e+05
max      1.111111e+09
Name: Price, dtype: float64

In [33]:
#Keep realistic prices only: 10,000 to 1,500,000 MAD
df_clean = df_clean[(df_clean["Price"] >= 10_000) & (df_clean["Price"] <= 1_500_000)]

In [34]:
#Remove placeholder prices made of one repeated digit (like 99999 or 1111111)
placeholder = df_clean["Price"].astype(int).astype(str).str.fullmatch(r"(\d)\1{3,}")
print(placeholder.sum(), "placeholder prices removed")
df_clean = df_clean[~placeholder]

19 placeholder prices removed


In [35]:
#Remove duplicate listings
print(df_clean.duplicated().sum(), "duplicate rows")
df_clean = df_clean.drop_duplicates()
df_clean.shape

1689 duplicate rows


(72476, 15)

Condition, Origin, First Owner and Number of Doors have missing values. Instead of dropping those rows, we mark the value as "Unknown" so we keep the data.

In [36]:
for col in ["Condition", "Origin", "First Owner"]:
    df_clean[col] = df_clean[col].fillna("Unknown")

df_clean["Number of Doors"] = df_clean["Number of Doors"].map(
    lambda v: "Unknown" if pd.isna(v) else str(int(v))
)

In [37]:
df_clean.isnull().sum()

Brand                0
Model                0
Year                 0
Condition            0
Mileage              0
Gearbox              0
Fiscal Power         0
Fuel                 0
Equipment            0
Number of Doors      0
Origin               0
First Owner          0
Location             0
Sector             166
Price                0
dtype: int64

In [38]:
selected_cols = ["Year", "Mileage", "Fiscal Power", "Price", "Brand", "Model", "Fuel", "Gearbox", "Location",
                 "Condition", "Origin", "First Owner", "Number of Doors"]
df_model = df_clean[selected_cols].copy()

df_model.head()

,Year,Mileage,Fiscal Power,Price,Brand,Model,Fuel,Gearbox,Location,Condition,Origin,First Owner,Number of Doors
0,1997,224999,8,27000.0,Volkswagen,Polo,Diesel,Manual,Tétouan,Good,Unknown,Unknown,Unknown
1,2012,184999,6,69000.0,Dacia,Sandero,Diesel,Manual,Temara,Very Good,WW in Morocco,No,5
2,2004,324999,8,45000.0,Volkswagen,Polo,Diesel,Manual,Safi,Fair,WW in Morocco,No,5
3,2009,47499,34,720000.0,Bentley,Continental Gtc,Petrol,Automatic,Casablanca,Excellent,Customs-cleared car,Yes,3
4,2013,87499,8,128000.0,Mini,Cooper,Diesel,Automatic,Fès,Excellent,WW in Morocco,No,5


In [39]:
#Making sure there are no null values in the final dataset
df_model.isnull().sum() 

Year               0
Mileage            0
Fiscal Power       0
Price              0
Brand              0
Model              0
Fuel               0
Gearbox            0
Location           0
Condition          0
Origin             0
First Owner        0
Number of Doors    0
dtype: int64

In [40]:


df_model["Brand"].value_counts().tail(20)

Brand
76                15
Zotye             13
Foton              9
Infiniti           9
Iveco              9
Subaru             8
77                 8
Bentley            7
Hummer             5
74                 4
Ac                 4
Byd                4
112                4
111                3
Ufo                3
Gmc                3
99                 2
Masey Ferguson     2
Force              1
72                 1
Name: count, dtype: int64

In [41]:
#Removing brands with numerical names
df_model = df_model[~df_model["Brand"].str.isnumeric()]

Dataset cleaning & selection of main features is done, now we need to encode categorical features using one-hot encoding because it converts categories into binary entries, useful for math operations like cosine similarity, distance etc..


In [42]:
# Seperating the features and target variable
X = df_model.drop("Price", axis=1)  
y = df_model["Price"]

# Splitting BEFORE encoding, so nothing from the test set influences the training features
# We will use 80% of the data for training and 20% for testing.
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [43]:
# creating one-hot encoded features for categorical variables
# The encoder lives inside a pipeline, so it learns the categories from the training set only.
# min_frequency=20 groups very rare categories together (this replaces the 1,400 columns we had before),
# and handle_unknown="ignore" lets the model accept brands or models it has never seen.
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline

categorical_features = ["Brand", "Model", "Fuel", "Gearbox", "Location",
                        "Condition", "Origin", "First Owner", "Number of Doors"]
numeric_features = ["Year", "Mileage", "Fiscal Power"]

preprocessor = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore", min_frequency=20), categorical_features),
    ("num", StandardScaler(), numeric_features),
])

def make_model(regressor):
    # The price is modelled on a log scale, so a few very expensive cars do not dominate the training
    pipe = Pipeline([("preprocess", preprocessor), ("regressor", regressor)])
    return TransformedTargetRegressor(regressor=pipe, func=np.log1p, inverse_func=np.expm1)

print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

(57892, 12)
(14474, 12)
(57892,)
(14474,)


**4. Training Models**

In [44]:
#The first model is Random Forest Regressor, because it is suitable for numerical features and is less prone to overfitting
from sklearn.ensemble import RandomForestRegressor

model = make_model(RandomForestRegressor(
    n_estimators=100, # 100 Trees
    min_samples_leaf=2, # each leaf needs at least 2 listings, which keeps the model smaller
    random_state=42, # For reproducibility
    n_jobs=-1 # Use all available cores to train faster
))

model.fit(X_train, y_train)

,"regressor regressor: object, default=NoneRegressor object such as derived from:class:`~sklearn.base.RegressorMixin`. This regressor willautomatically be cloned each time prior to fitting. If `regressor isNone`, :class:`~sklearn.linear_model.LinearRegression` is created and used.",Pipeline(step...m_state=42))])
,"func func: function, default=NoneFunction to apply to `y` before passing to :meth:`fit`. Cannot be setat the same time as `transformer`. If `func is None`, the function used will bethe identity function. If `func` is set, `inverse_func` also needs to beprovided. The function needs to return a 2-dimensional array.",<ufunc 'log1p'>
,"inverse_func inverse_func: function, default=NoneFunction to apply to the prediction of the regressor. Cannot be set atthe same time as `transformer`. The inverse function is used to returnpredictions to the same space of the original training labels. If`inverse_func` is set, `func` also needs to be provided. The inversefunction needs to return a 2-dimensional array.",<ufunc 'expm1'>
,"transformer transformer: object, default=NoneEstimator object such as derived from:class:`~sklearn.base.TransformerMixin`. Cannot be set at the same timeas `func` and `inverse_func`. If `transformer is None` as well as`func` and `inverse_func`, the transformer will be an identitytransformer. Note that the transformer will be cloned during fitting.Also, the transformer is restricting `y` to be a numpy array.",None
,"check_inverse check_inverse: bool, default=TrueWhether to check that `transform` followed by `inverse_transform`or `func` followed by `inverse_func` leads to the original targets.",True
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Defined only when `X`has feature names that are all strings... versionadded:: 1.0","ndarray[object](12,)","['Year','Mileage','Fiscal Power',...,'Origin','First Owner', 'Number of Doors']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying regressor exposes such an attribute when fit... versionadded:: 0.24,int,12
regressor_ regressor_: objectFitted regressor.,Pipeline,Pipeline(step...m_state=42))])
transformer_ transformer_: objectTransformer used in :meth:`fit` and :meth:`predict`.,FunctionTransformer,FunctionTrans...validate=True)
,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocess', ...), ('regressor', ...)]"


In [45]:
#The second model is Linear Regression, a simple baseline to compare against
from sklearn.linear_model import LinearRegression

model_lr = make_model(LinearRegression())
model_lr.fit(X_train, y_train)

,"regressor regressor: object, default=NoneRegressor object such as derived from:class:`~sklearn.base.RegressorMixin`. This regressor willautomatically be cloned each time prior to fitting. If `regressor isNone`, :class:`~sklearn.linear_model.LinearRegression` is created and used.",Pipeline(step...egression())])
,"func func: function, default=NoneFunction to apply to `y` before passing to :meth:`fit`. Cannot be setat the same time as `transformer`. If `func is None`, the function used will bethe identity function. If `func` is set, `inverse_func` also needs to beprovided. The function needs to return a 2-dimensional array.",<ufunc 'log1p'>
,"inverse_func inverse_func: function, default=NoneFunction to apply to the prediction of the regressor. Cannot be set atthe same time as `transformer`. The inverse function is used to returnpredictions to the same space of the original training labels. If`inverse_func` is set, `func` also needs to be provided. The inversefunction needs to return a 2-dimensional array.",<ufunc 'expm1'>
,"transformer transformer: object, default=NoneEstimator object such as derived from:class:`~sklearn.base.TransformerMixin`. Cannot be set at the same timeas `func` and `inverse_func`. If `transformer is None` as well as`func` and `inverse_func`, the transformer will be an identitytransformer. Note that the transformer will be cloned during fitting.Also, the transformer is restricting `y` to be a numpy array.",None
,"check_inverse check_inverse: bool, default=TrueWhether to check that `transform` followed by `inverse_transform`or `func` followed by `inverse_func` leads to the original targets.",True
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Defined only when `X`has feature names that are all strings... versionadded:: 1.0","ndarray[object](12,)","['Year','Mileage','Fiscal Power',...,'Origin','First Owner', 'Number of Doors']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying regressor exposes such an attribute when fit... versionadded:: 0.24,int,12
regressor_ regressor_: objectFitted regressor.,Pipeline,Pipeline(step...egression())])
transformer_ transformer_: objectTransformer used in :meth:`fit` and :meth:`predict`.,FunctionTransformer,FunctionTrans...validate=True)
,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocess', ...), ('regressor', ...)]"


In [46]:
#The third model is a Decision Tree
from sklearn.tree import DecisionTreeRegressor

model_dt = make_model(DecisionTreeRegressor(min_samples_leaf=5, random_state=42))
model_dt.fit(X_train, y_train)

,"regressor regressor: object, default=NoneRegressor object such as derived from:class:`~sklearn.base.RegressorMixin`. This regressor willautomatically be cloned each time prior to fitting. If `regressor isNone`, :class:`~sklearn.linear_model.LinearRegression` is created and used.",Pipeline(step...m_state=42))])
,"func func: function, default=NoneFunction to apply to `y` before passing to :meth:`fit`. Cannot be setat the same time as `transformer`. If `func is None`, the function used will bethe identity function. If `func` is set, `inverse_func` also needs to beprovided. The function needs to return a 2-dimensional array.",<ufunc 'log1p'>
,"inverse_func inverse_func: function, default=NoneFunction to apply to the prediction of the regressor. Cannot be set atthe same time as `transformer`. The inverse function is used to returnpredictions to the same space of the original training labels. If`inverse_func` is set, `func` also needs to be provided. The inversefunction needs to return a 2-dimensional array.",<ufunc 'expm1'>
,"transformer transformer: object, default=NoneEstimator object such as derived from:class:`~sklearn.base.TransformerMixin`. Cannot be set at the same timeas `func` and `inverse_func`. If `transformer is None` as well as`func` and `inverse_func`, the transformer will be an identitytransformer. Note that the transformer will be cloned during fitting.Also, the transformer is restricting `y` to be a numpy array.",None
,"check_inverse check_inverse: bool, default=TrueWhether to check that `transform` followed by `inverse_transform`or `func` followed by `inverse_func` leads to the original targets.",True
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Defined only when `X`has feature names that are all strings... versionadded:: 1.0","ndarray[object](12,)","['Year','Mileage','Fiscal Power',...,'Origin','First Owner', 'Number of Doors']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying regressor exposes such an attribute when fit... versionadded:: 0.24,int,12
regressor_ regressor_: objectFitted regressor.,Pipeline,Pipeline(step...m_state=42))])
transformer_ transformer_: objectTransformer used in :meth:`fit` and :meth:`predict`.,FunctionTransformer,FunctionTrans...validate=True)
,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocess', ...), ('regressor', ...)]"


**5. Testing**

In [47]:
y_pred = model.predict(X_test)
y_pred[:5]

array([162698.53692104, 113084.21923794,  70596.8272855 , 173964.26026892,
        67938.57510844])

In [48]:
y_test.head()

92498    170000.0
89429    120000.0
36924     80000.0
60435    120000.0
45043     56000.0
Name: Price, dtype: float64

In [49]:
y_pred_lr = model_lr.predict(X_test)
y_pred_dt = model_dt.predict(X_test)

**6. Evaluating Results**

In [50]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

In [51]:
def evaluate(y_true, y_pred):
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "R2": r2_score(y_true, y_pred),
        "Median % error": np.median(np.abs(y_pred - y_true) / y_true) * 100,
    }

results = pd.DataFrame({
    "Linear Regression": evaluate(y_test, y_pred_lr),
    "Decision Tree": evaluate(y_test, y_pred_dt),
    "Random Forest": evaluate(y_test, y_pred),
}).T
results.round(3)

,MAE,RMSE,R2,Median % error
Linear Regression,25547.659,74594.525,0.652,10.690
Decision Tree,23350.522,70564.475,0.689,9.474
Random Forest,18763.079,66014.428,0.728,7.162


The most important metric for this project is the MAE, because it tells the user how far off the estimate is in MAD on average. The Random Forest has the lowest MAE, so it is the model we use.

**7. Fair Price Range and Verdict**

- A single predicted price is not enough, because real prices vary around it. We look at how far the real prices were from the predictions on the test set, and use the middle 80% of that spread as the "fair range". A seller price below the range is underpriced, above it is overpriced.

In [52]:
ratio = y_test.to_numpy() / y_pred   # real price divided by predicted price
ratio_low, ratio_high = np.quantile(ratio, [0.10, 0.90])
print(f"Fair range = predicted price x [{ratio_low:.2f}, {ratio_high:.2f}]")

def check_listing(predicted, seller_price):
    low, high = predicted * ratio_low, predicted * ratio_high
    if seller_price < low:
        return "Underpriced"
    if seller_price > high:
        return "Overpriced"
    return "Fair"

Fair range = predicted price x [0.82, 1.19]


In [53]:
# Examples: the real listing prices, then the same cars priced 60% higher
examples = X_test.head(5)[["Brand", "Model", "Year"]].copy()
examples["Predicted"] = y_pred[:5].round(0)
examples["Seller price"] = y_test.head(5).values
examples["Verdict"] = [check_listing(p, s) for p, s in zip(y_pred[:5], y_test.head(5))]
examples["Seller price (+60%)"] = (y_test.head(5).values * 1.6).round(0)
examples["Verdict (+60%)"] = [check_listing(p, s * 1.6) for p, s in zip(y_pred[:5], y_test.head(5))]
examples

,Brand,Model,Year,Predicted,Seller price,Verdict,Seller price (+60%),Verdict (+60%)
92498,Renault,Megane_4,2019,162699.0,170000.0,Fair,272000.0,Overpriced
89429,Mercedes-Benz,Classe Cls,2008,113084.0,120000.0,Fair,192000.0,Overpriced
36924,Renault,Kangoo,2012,70597.0,80000.0,Fair,128000.0,Overpriced
60435,Ford,Focus,2014,173964.0,120000.0,Underpriced,192000.0,Fair
45043,Dacia,Logan,2012,67939.0,56000.0,Fair,89600.0,Overpriced


**8. Saving the Model**

- The final model used by the terminal tool is trained and saved by `src/train.py`, which repeats the cleaning and training steps above. To use it, run `python src/train.py` once, then `python src/predict.py`.